# Do earthquakes cluster — or is that just what randomness looks like?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2FAI4EPS%2FEPS88_PyEarth&branch=main&urlpath=lab%2Ftree%2FEPS88_PyEarth%2Fdocs/notebooks%2F05_clustered_or_random.ipynb).

On a map, earthquakes obviously cluster: they draw the plate boundaries, which is what the first
week of this course found. In time it is much harder to tell. Plot every large earthquake of the
last twenty-six years day by day and the record looks bunched — long quiet stretches, then several
in a week — and it is tempting to say so and move on.

The trouble is that bunching is what randomness does. Scatter twelve thousand earthquakes at
random across nine thousand days and you get quiet stretches and busy weeks too, because *random*
does not mean *evenly spread*. So the eye cannot settle this, and no stronger adjective will
either.

Today you build the world where earthquakes happen at random at the real rate, measure how clumpy
that world gets, and compare it with the one we live on. Whatever is left over is physics.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by an empty cell. Fill them all in, then export the notebook as a PDF and upload that.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## What you'll be able to do

**The science.** Say whether earthquakes really are more bunched in time than chance would make
them, by how much, and what the excess is made of. Then turn a catalogue into a forecast: the
chance of an earthquake near campus in the next few years, and what that number quietly assumes.

**The skills.** Dates that Python understands: `pd.to_datetime`, `set_index`, and `resample` to
count events per day. Simulation: `np.random.default_rng` and a seed, so a random experiment gives
the same answer twice, and a loop that runs it two thousand times. And the Poisson formula,
`np.exp`, which gets the same answers without the simulation.

**Eight places where you write something: five in class, three at home.** Each one is headed
*Your turn*, with an empty cell under it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.figsize": (7, 3.6), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

def load(name, start, end, minmag, region):
    """Fetch one window of the USGS catalogue; fall back to the copy stored with the course.

    `region` is extra query text limiting the search to a box of latitude and longitude;
    pass "" for the whole world.
    """
    try:
        return pd.read_csv(f"https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv&orderby=time-asc"
                       f"&starttime={start}&endtime={end}&minmagnitude={minmag}"
                       + region)
    except Exception as e:
        print("live source unreachable, using the cached copy:", type(e).__name__)
        return pd.read_csv(CACHE + "/" + f"week05_{name}_{start}_{end}_M{minmag}.csv")

# Two degrees of latitude and longitude each side of Berkeley, which sits at
# 37.87 north, -122.27 east. You need this one at the end of the notebook.
BAY_BOX = ("&minlatitude=35.87&maxlatitude=39.87"
           "&minlongitude=-124.27&maxlongitude=-120.27")

quakes = load("global", "2000-01-01", "2026-01-01", 5.5, "")
bay = load("bay", "1900-01-01", "2026-01-01", 5.0, BAY_BOX)
print("global catalogue:", quakes.shape, " Bay Area catalogue:", bay.shape)

## From a list of earthquakes to a count per day

`quakes` holds every earthquake of magnitude 5.5 and above that the USGS recorded worldwide
between the start of 2000 and the start of 2026. One row is one earthquake, and the
`time` column says when — but only as far as pandas is concerned it is text, and you cannot count
text by the day.

`pd.to_datetime` turns that column into real dates. `set_index` then makes the time column the row
labels, so the table knows *when* each row happened rather than just holding a column about it.
Once it does, `resample("D")` regroups the rows into fixed windows — `"D"` for one day each — and
`.count()` says how many fell in each.

In [ ]:
quakes["time"] = pd.to_datetime(quakes["time"])
daily = quakes.set_index("time")["mag"].resample("D").count()

print(daily.head())
print("days covered:      ", len(daily))
print("earthquakes:       ", daily.sum())
print("average per day:   ", round(daily.mean(), 3))

12,849 earthquakes over 9,497 days, so a bit over one a day on average. Every
day is in there, including the ones with nothing on them — that is what `resample` gives you and
it is what we want, because a day with no earthquake is data too.

Now draw it. One point per day, 9,497 of them.

In [ ]:
plt.plot(daily.index, daily.values, lw=0.5)
plt.xlabel("date")
plt.ylabel("earthquakes M5.5+ per day")
plt.title("12,849 earthquakes over 9,497 days")
plt.show()

### ✏️ Your turn 1

One day in that plot towers over the rest. Which day was it, and how many earthquakes did it hold?

`daily.max()` gives the largest count. `daily.idxmax()` gives the *label* of the largest value —
here, the date — the way `.argmax()` gave you the position of the largest cell in a grid.
`str(...)[:10]` trims the timestamp down to just the date.

**Use these names**, because the self-check looks for them: `busiest_count`, `busiest_day`.

In [ ]:
# ← your answer here


assert "-" in busiest_day, \
    "busiest_day should read like a date — .idxmax() gives the label, .argmax() gives a row number"
print("✓ the busiest day —", busiest_day, "with", busiest_count, "earthquakes, against",
      round(daily.mean(), 3), "on an average day")

## What a world without clustering looks like

128 in one day against 1.353 on an average day is a startling ratio, and the
temptation is to declare the catalogue clustered and stop. Resist it for one section, because you
have nothing to compare against. Startling *compared with what?*

So build the comparison. Take the same 12,849 earthquakes and the same
9,497 days, and throw each earthquake onto a day picked at random. That is a world with
the same rate as ours and no clustering of any kind, because nothing in it knows about anything
else. Then count its days the way you counted the real ones.

`np.random.default_rng(seed)` makes a random-number generator. The `seed` is the only unusual part:
a computer's random numbers are produced by a recipe, and the seed is where the recipe starts, so
the same seed gives the same "random" numbers every time. That is what makes a simulation
something you can hand to somebody else. `rng.integers(0, n_days, n_quakes)` then draws that many
whole numbers between 0 and `n_days`, one day number per earthquake, and `np.histogram` with one
bin per day counts how many landed on each.

In [ ]:
n_days = len(daily)
n_quakes = len(quakes)
day_edges = np.arange(n_days + 1)          # one bin per day


def random_world(rng):
    """Scatter n_quakes earthquakes at random over n_days days, and count each day."""
    day_numbers = rng.integers(0, n_days, n_quakes)
    counts, edges = np.histogram(day_numbers, bins=day_edges)
    return counts

### Predict before you run

In that world nothing is clustered — every earthquake landed on a day chosen by a coin the
universe has no memory of. Out of 9,497 such days, how many earthquakes do you think the
busiest single one will hold? The average day holds 1.353.

Change `my_guess` to whatever you think, then run the cell.

In [ ]:
my_guess = 3

rng = np.random.default_rng(88)
one_world = random_world(rng)

print("you guessed:                   ", my_guess)
print("the random world's busiest day:", one_world.max())

8, from a process with no clustering in it at all. If you guessed three or four
you are in good company, and the reason is worth holding on to: 12,849 things scattered
over 9,497 days will not lay themselves out one-per-day-and-a-bit. Some days get none,
some get five, and out of nine and a half thousand tries something is going to get
8.

Here is the same point as a picture. The first two hundred days of the real catalogue, then the
first two hundred days of the random world, on the same axes.

In [ ]:
plt.plot(daily.index[:200], daily.values[:200], lw=1)
plt.ylim(0, 9)
plt.xlabel("date")
plt.ylabel("earthquakes M5.5+ per day")
plt.title("the real catalogue, 200 days")
plt.show()

plt.plot(daily.index[:200], one_world[:200], lw=1)
plt.ylim(0, 9)
plt.xlabel("date")
plt.ylabel("earthquakes M5.5+ per day")
plt.title("a random world, the same 200 days")
plt.show()

Cover the titles and you would struggle to say which is which. Both have runs of empty days, both
have spikes, both have stretches that look busier than the rest. **Randomness is already clumpy, so
"it looks clustered" proves nothing on its own** — and that is not a lesson about earthquakes, it
is a lesson about eyes.

### ✏️ Your turn 2

One random world is an anecdote. Run four more, with four seeds of your own choosing — your
birthday, the last digits of your student ID, anything — and collect the busiest day of each.

Build the list with the accumulator pattern: start `busiest_by_seed` empty, then inside a `for`
loop over your seeds, make a generator with `np.random.default_rng(seed)`, call
`random_world(rng)`, and append `int(...)` of `.max()` of what comes back — `int` because numpy's
own whole numbers print with their type attached, which is noise here.

**Use these names**, because the self-check looks for them: `my_seeds`, `busiest_by_seed`.

In [ ]:
# ← your answer here


assert len(busiest_by_seed) == len(my_seeds), \
    "one busiest day per seed, so the two lists should be the same length"
assert max(busiest_by_seed) < daily.max(), \
    "a random world reached the real busiest day — check that you called random_world"
print("✓ four more random worlds — busiest days", busiest_by_seed,
      "against", daily.max(), "in the real catalogue")

## Two thousand worlds instead of one

Five worlds is better than one and still not an answer. What you want is the whole *range* of
busiest days a world without clustering produces, so that you can say where the real catalogue
falls in it. That is a **Monte Carlo** simulation: Make up a world where the effect is absent, a
thousand times, and see how often chance alone beats what you measured.

You already have the machinery. Run `random_world` 2,000 times and keep only the number you
care about each time — the busiest day.

In [ ]:
rng = np.random.default_rng(88)
busiest_days = []

for run in range(2000):
    busiest_days.append(random_world(rng).max())

print("worlds simulated:", len(busiest_days))

In [ ]:
plt.hist(busiest_days, bins=np.arange(min(busiest_days) - 0.5, 128 + 5))
plt.axvline(daily.max(), color="firebrick")
plt.xlabel("busiest single day of a whole world (earthquakes M5.5+)")
plt.ylabel("number of simulated worlds")
plt.title("2,000 worlds without clustering; the red line is the real catalogue")
plt.show()

Every one of the 2,000 simulated worlds is in that little pile on the left. The red line is
where the catalogue we actually live in sits. There is nothing in between.

That figure is the week's argument in one picture, but a picture is not a number. Put numbers on it.

### ✏️ Your turn 3

`busiest_days` is a plain Python list. `np.array(...)` turns it into an array so that you can ask
it array questions.

Print four things: the mean of the 2,000 busiest days, their 95th percentile
(`np.percentile(worlds, 95)` — the value 95% of them fall below), how many of the 2,000
worlds reached the real busiest day, and the real busiest day divided by the simulated mean.

**Use these names**, because the self-check looks for them: `worlds`.

In [ ]:
# ← your answer here


assert worlds.max() < daily.max(), \
    "if a simulated world reached the real busiest day, something is wrong with the simulation"
print("✓ the comparison — the busiest day of a random world averages",
      round(worlds.mean(), 2), "and the real one is", daily.max(), "—",
      round(daily.max() / worlds.mean(), 1), "times larger")

## The day itself, and the year around it

A factor of 16.7 is not a near miss. Nothing chance produces looks like the catalogue,
so something other than chance is putting earthquakes on the same day as each other. The catalogue
can say what.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
quakes["time"] = pd.to_datetime(quakes["time"])
daily = quakes.set_index("time")["mag"].resample("D").count()

In [ ]:
day = quakes[(quakes["time"] >= "2011-03-11") & (quakes["time"] < "2011-03-12")]

print(len(day), "earthquakes on", "2011-03-11")
print(day.sort_values("mag", ascending=False).head(1)[["latitude", "longitude", "mag", "place"]])

In [ ]:
close = (abs(day["latitude"] - 38.3) < 5) & (abs(day["longitude"] - 142.4) < 5)

print(close.sum(), "of the", len(day), "were within 5 degrees of that one")

Magnitude 9.1, off the Pacific coast of northern Japan — the largest earthquake in this
catalogue. And 127 of the day's 128 earthquakes happened within five
degrees of it.

That is an **aftershock** sequence. A large earthquake does not release the stress in the crust
tidily: it slips over a patch of fault hundreds of kilometres long, and in doing so it loads the
rock around the edges of that patch and on every neighbouring fault. Those places then fail in
turn — over hours, then weeks, then years, at a rate that dies away. So the events are not
independent. One earthquake makes the next one more likely, in a particular place and at a
particular time, and that is exactly the assumption the random world was built without.

How long does it last? `daily.loc[a:b]` reads out a stretch of days between two dates, and
`.sum()` adds them up.

In [ ]:
print("earthquakes in the 30 days from 2011-03-11:", daily.loc["2011-03-11":"2011-04-09"].sum())
print("earthquakes in the 30 days before it: ", daily.loc["2011-02-09":"2011-03-10"].sum())

223 against 52 — the month after the mainshock held about four times the
month before it. So the excess is not confined to one day.

One more thing worth checking before drawing a conclusion, because the simulation assumed the rate
was the same every day for twenty-six years. Was it? `rolling(365).sum()` slides a 365-day window
along the series and adds up what is inside it, which turns a spiky daily count into a running
year.

In [ ]:
running = daily.rolling(365).sum()

plt.plot(running.index, running.values)
plt.xlabel("date")
plt.ylabel("earthquakes M5.5+ in the previous 365 days")
plt.title("a running year, 9,497 daily windows")
plt.show()

print("lowest running year: ", running.min())
print("highest running year:", running.max(), "on", str(running.idxmax())[:10])
print("median:              ", running.median())

The rate is not a constant: a running year holds anywhere from 335 to 743
earthquakes around a median of 477. But there is no long climb across the record, and
the highest running year of all ends on 2011-09-22 — which is to say the biggest wobble in
the rate is the same aftershock sequence, seen through a wider window. A flat rate is an
approximation, and a fair one for this comparison, because the simulation was handed exactly the
12,849 earthquakes the catalogue holds.

Which leaves the obvious worry: is this whole result one earthquake in Japan?

### ✏️ Your turn 4

Answer it with the neighbouring windows. `daily.index.year` gives the year of every day in the
series, so `daily.groupby(daily.index.year).max()` splits the 26 years apart and gives
you the busiest day of each.

Build that, draw it with `plt.scatter`, and put a horizontal line at the 95th percentile you
printed above with `plt.axhline(9, color="firebrick")` — that is the bar a world
without clustering clears only one time in twenty. Then print the series itself, and count how many
years are above the line.

**Use these names**, because the self-check looks for them: `by_year`.

In [ ]:
# ← your answer here


assert len(by_year) == 26, "one number per year, so there should be 26"
print("✓ every year, not one —", (by_year > 9).sum(), "of 26 years hold a day above",
      9, "and", (by_year > 9).sum() - 1, "of those are not 2011")

18 of the 26 years clear a bar that a world without clustering clears
once in twenty, and even the quietest year in the record reaches 6. Take
2011 out entirely and 17 years still do it. This is not one earthquake in Japan;
it is what the record looks like everywhere you cut it.

## A formula instead of a simulation

Simulating 2,000 worlds took a moment and a screenful of code. For some questions about a
world with no clustering there is a formula that gives the answer exactly, with no simulation at
all, and it is worth having because it is what a real forecast is built from.

If events happen at random at an average rate of λ per interval, then the chance that a given
interval holds **none at all** is e to the power of −λ — in Python, `np.exp(-lam)`. That is the
**Poisson** formula, and λ is the only thing it needs. The simulation and the formula are two
routes to the same place: one *empirical*, got by counting what happened, and one *theoretical*,
got by evaluating an expression. If they disagree, one of them is wrong.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
quakes["time"] = pd.to_datetime(quakes["time"])
daily = quakes.set_index("time")["mag"].resample("D").count()
n_days, n_quakes = len(daily), len(quakes)
day_edges = np.arange(n_days + 1)


def random_world(rng):
    """Scatter n_quakes earthquakes at random over n_days days, and count each day."""
    counts, edges = np.histogram(rng.integers(0, n_days, n_quakes), bins=day_edges)
    return counts


one_world = random_world(np.random.default_rng(88))

In [ ]:
lam = daily.mean()

print("rate, earthquakes per day:      ", round(lam, 3))
print("formula says quiet days:        ", round(np.exp(-lam), 4))
print("the simulated world actually had:", round((one_world == 0).mean(), 4))

0.2585 against 0.2505. The formula and the simulation agree, which is
the point: the simulation was never doing anything magical, it was drawing from the world the
formula describes.

Two more things fall straight out of it. The chance of **at least one** event in an interval is
whatever is left over, `1 - np.exp(-lam)`. And λ can be scaled to any interval you like: a day is
1.353 earthquakes, so an hour is λ divided by 24. Finally, `1 / lam` is the average wait
between events — the **recurrence interval**.

In [ ]:
print("chance of at least one somewhere on Earth today:      ", round(1 - np.exp(-lam), 4))
print("chance of at least one in the next hour:              ", round(1 - np.exp(-lam / 24), 4))
print("average hours between them:                           ", round(24 / lam, 1))

### ✏️ Your turn 5

The formula describes a world without clustering. The catalogue is not that world, and the busiest
day already showed one way it differs. Here is the other end of the same distribution.

Count the days in the real catalogue that held no earthquake at all — `(daily == 0).sum()` — and
work out how many the formula expects, which is `np.exp(-lam) * n_days`. Print both, and print the
difference.

**Use these names**, because the self-check looks for them: `real_quiet`, `expected_quiet`.

In [ ]:
# ← your answer here


assert real_quiet > 1, "real_quiet should be a COUNT of days, not a fraction — .sum(), not .mean()"
print("✓ quiet days —", real_quiet, "in the catalogue against", round(expected_quiet),
      "from the formula:", real_quiet - round(expected_quiet), "more silence than chance allows")

So the real Earth has 550 *more* empty days than a world without clustering, as
well as a day holding 128. Both at once, and they are the same fact: piling earthquakes
into a few days has to leave other days emptier, because the total is fixed. That is what
clustering does to a record, at both ends.

## The question, answered

**They cluster, and it is not an illusion — but you could not have known that by looking.** A world
with no clustering at all, running at Earth's own rate, produces a busiest day of about
7.7 and never once in 2,000 tries got past 11; the real
catalogue's busiest day holds 128, about 16.7 times chance, and
18 of 26 separate years break the same bar. The excess is aftershocks:
127 of the 128 earthquakes on 2011-03-11 were within five
degrees of a single magnitude-9.1 rupture, and the month after it held 223
events against 52 in the month before. That is physics — stress transferred to
neighbouring rock, which fails in its turn — not noise.

What did the work was not a cleverer look at the data. It was building the world where the effect
is absent and measuring how often chance alone beats what you measured.

## Week 5 summary

**The question.** Do earthquakes cluster — or is that just what randomness looks like?

### What to remember

| | |
|---|---|
| **1** | Randomness is already clumpy, so "it looks clustered" proves nothing on its own. |
| **2** | Real seismicity still exceeds chance by a wide margin, and that excess is aftershocks — physics, not noise. |
| **3** | To test whether a pattern is real, simulate the world where it is absent and compare. |

### The ideas, in plain words

| Idea | Means |
|---|---|
| **Monte Carlo** | Make up a world where the effect is absent, a thousand times, and see how often chance alone beats what you measured. |

## Homework

Three parts, and a change of scale. Class asked whether earthquakes cluster; the homework asks the
question people actually want answered, which is what the chance is of one happening **here**,
while you are here.

`bay` is already loaded: every earthquake of magnitude 5.0 and above the USGS records within
two degrees of Berkeley, from 1900 to 2026 — 83 of them over
126 years. If you have restarted since class, run the setup cell at the top first. The
oldest events in it predate modern instruments, and their magnitudes were reconstructed later from
written accounts of the shaking, so treat them as approximate.

### ✏️ Your turn 6

Berkeley sits at latitude 37.87, longitude -122.27. So
`abs(bay["latitude"] - 37.87)` is how far north or south a row is from campus in degrees, and
`abs(bay["longitude"] - (-122.27))` is how far east or west.

Keep the rows within **1.0 degree** of campus in both directions, call that `near`, and from it
keep the rows of magnitude 6.0 and above, called `big`. Then:

- a **rate**: how many of them per year, over the 126 years the query covers;
- a **recurrence interval**: `1 / rate`, the average wait in years;
- and the chance of **at least one** during four years at Berkeley, `1 - np.exp(-rate * 4)`.

Print all four.

**Use these names**, because the self-check looks for them: `near`, `big`, `rate`.

In [ ]:
# ← your answer here


assert len(big) < len(near), "big is the magnitude-6 part of near, so it must be the smaller one"
assert rate < 1, "that is a rate per YEAR — above 1 means you divided by the wrong number"
print("✓ the four-year forecast —", len(big), "earthquakes,", round(1 / rate, 1),
      f"years apart on average, and a {round(100 * (1 - np.exp(-rate * 4)))}% chance in four years")

### ✏️ Your turn 7

"Near Berkeley" was your choice, and it changed the answer. Find out by how much.

Write a function `forecast(half)` that does everything part 1 did, for a box `half` degrees on each
side of campus, and **returns** the four-year chance. Inside it, also print:

- the events themselves — `print(big[["time", "mag", "place"]])`;
- the gaps between them in days. `pd.to_datetime(big["time"]).diff()` gives the interval between
  each event and the one before it — the first is blank, so `.dropna()` — and `.dt.days` turns each
  interval into a whole number of days. `sorted(gaps)` then puts them in order, shortest first.
  Sort `big` by `"time"` first, or the gaps are meaningless;
- and the chance of at least one in **30 years**, which is the window published earthquake
  forecasts conventionally use.

Then call it twice, at `half = 1.0` and `half = 2.0`, keeping the two answers.

**Use these names**, because the self-check looks for them: `forecast`, `p1`, `p2`.

In [ ]:
# ← your answer here


assert p2 > p1, "the bigger box holds more earthquakes, so its four-year chance must be the larger"
print(f"✓ the two boxes — {round(100 * p1)}% within one degree of campus "
      f"and {round(100 * p2)}% within two")

### ✏️ Your turn 8

You now have two numbers for the same question and, under each of them, the earthquakes they were
built from and the gaps between those earthquakes. Three or four sentences, using **your own
printed output**:

1. Which of the two four-year numbers would you quote, and what does the bigger box buy and cost?
   Name at least one earthquake it adds.
2. The Poisson formula assumes events arrive independently at a steady rate. Quote your shortest
   gap and your recurrence interval, and say whether those two numbers are consistent with that
   assumption.
3. Given what class measured about clustering, say in which direction you distrust your number.

*(Double-click this cell and replace this line with your answer.)*